# UNIパッチ ベクトル検索 デモ

`experiments/0001_..._build_patch_manifest` → `0002_..._build_faiss_index` を実行済みであること前提。

1枚以上の参照画像(同じ所見の異なる撮影を複数渡せる)を渡すと、
1. 類似パッチ上位k件(スライドID+座標+コサイン類似度)
2. 類似パッチを多く含むWSIの逆引きランキング
3. 上位WSIのサムネイル上に類似パッチ位置を可視化

を返す `lib/search.py::PatchIndex` を対話的に試す。

**推奨パイプライン(このノートブックの既定)**: `lib.query_embedding.embed_image_tiles`
(タイル分割、倍率補正・染色正規化なし) + `PatchIndex.search_similar_patches_multi` /
`search_top_slides_multi`。単一クロップ(`embed_image`)は病変の位置が分からない
大きな参照画像では不利(タイル分割の方が一貫して優れる)。倍率自動補正
(`embed_image_tiles_auto_scale`)とMacenko染色正規化(`stain_reference=`)は
どちらも実装済みで動くが、7カテゴリのground truth比較で**検索精度を悪化させる
ことが確認済み**なので、既定では使わないこと — 使う前に
`scripts/validate_against_ground_truth.py`で自分のケースについて検証すること。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from lib.query_embedding import embed_image_tiles
from lib.search import PatchIndex
from lib.visualize import plot_slide_hits_on_thumbnail

In [ ]:
INDEX_EXP_DIR = PROJECT_ROOT / "outputs/0002_20260808_build_faiss_index/default"
FEATURES_DIR = PROJECT_ROOT / "data/trident_processed/20x_224px_0px_overlap/features_uni_v1"
THUMBNAILS_DIR = PROJECT_ROOT / "data/trident_processed/thumbnails"

patch_index = PatchIndex.load(
    index_path=INDEX_EXP_DIR / "index.faiss",
    manifest_path=INDEX_EXP_DIR / "manifest.parquet",
    slide_meta_path=INDEX_EXP_DIR / "slide_meta.parquet",
    features_dir=FEATURES_DIR,
)
patch_index.index.ntotal

In [ ]:
QUERY_IMAGES = ["/path/to/query_image.jpg"]  # ここを差し替える。同じ所見の複数画像を渡してOK

# 各画像をタイル分割して埋め込み、全部まとめて1つのクエリ集合にする
# (ベクトル平均ではなく個別タイル+結果統合の方が頑健 — 倍率の異なる画像を混ぜても
# 表現がぼやけない。詳細は lib/search.py::search_top_slides_multi のdocstring)
query_vecs = np.concatenate([embed_image_tiles(img) for img in QUERY_IMAGES], axis=0)
query_vecs.shape

## 1. 類似パッチ検索

In [ ]:
similar_patches = patch_index.search_similar_patches_multi(
    query_vecs, k=20, nprobe=32, rerank_pool=200, max_tiles_reranked=12
)
similar_patches

## 2. WSI逆引き(類似パッチを多く含むスライド)

`n_hits_ratio`(=n_hits/total_patches)で降順ソートされる(7カテゴリのground truth
比較で、単純な`n_hits`より5/7カテゴリで順位が改善することを確認済み — 大きいスライド
ほど絶対ヒット数を稼ぎやすいバイアスを補正するため)。

In [ ]:
top_slides = patch_index.search_top_slides_multi(query_vecs, k_candidates=8000, nprobe=64, top_n_slides=20)
top_slides

## 3. 上位スライドのサムネイル上に可視化

In [ ]:
# search_top_slides_multi drew from a much larger, non-reranked FAISS candidate
# pool (k_candidates=8000, nprobe=64) than search_similar_patches_multi's small
# exact-reranked rerank_pool=200 above, so a top slide often has zero rows
# in that smaller result. Re-derive a large exact-reranked pool with the
# SAME nprobe=64 as search_top_slides_multi (so its candidate set is a proper
# superset) once, and pull each plotted slide's hits from it. max_tiles_reranked
# stays small (not query_vecs.shape[0]) — reranking every tile at rerank_pool=8000
# each would be far more expensive than needed.
large_pool = patch_index.search_similar_patches_multi(
    query_vecs, k=8000, nprobe=64, rerank_pool=8000, max_tiles_reranked=12
)

for slide_id in top_slides["slide_id"].head(3):
    hits = similar_patches[similar_patches["slide_id"] == slide_id]
    if hits.empty:
        hits = large_pool[large_pool["slide_id"] == slide_id]
    if hits.empty:
        continue
    fig = plot_slide_hits_on_thumbnail(slide_id, hits, THUMBNAILS_DIR, patch_index.slide_meta)